### Tools in LangChain

Tools let the model call your own functions. This notebook shows:
1. Defining a simple tool with `@tool`
2. Tools with multiple arguments
3. Using a Pydantic schema for structured inputs
4. Binding tools directly to a model (`bind_tools`)
5. Giving an agent multiple tools

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Set all the environment variables
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Force API-key based Gemini route (not Vertex AI project auth)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

### 1. A simple tool

The `@tool` decorator turns any function into a tool. The docstring is sent to the model as the tool description, so keep it clear.

In [1]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

# Inspect what the model sees
print("name:", add.name)
print("description:", add.description)
print("args:", add.args)

# You can also call the tool directly
print("result:", add.invoke({"a": 2, "b": 3}))

name: add
description: Add two numbers together.
args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
result: 5


### 2. Tool with multiple arguments

Type hints become the tool's argument schema. Optional arguments get a default value.

In [2]:
from typing import Optional
from langchain_core.tools import tool

@tool
def get_weather(city: str, unit: Optional[str] = "celsius") -> str:
    """Get the current weather for a city. unit can be 'celsius' or 'fahrenheit'."""
    temp = 25 if unit == "celsius" else 77
    return f"The current weather in {city} is sunny with a temperature of {temp} degrees {unit}."

print(get_weather.invoke({"city": "Dhaka", "unit": "celsius"}))

The current weather in Dhaka is sunny with a temperature of 25 degrees celsius.


### 3. Structured input with a Pydantic schema

For richer validation and per-field descriptions, pass an `args_schema`. `Field` descriptions help the model fill arguments correctly.

In [3]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number")
    b: int = Field(description="The second number")

@tool(args_schema=MultiplyInput)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

print("args:", multiply.args)
print("result:", multiply.invoke({"a": 6, "b": 7}))

args: {'a': {'description': 'The first number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number', 'title': 'B', 'type': 'integer'}}
result: 42


### 4. Binding tools directly to a model

`bind_tools` lets the model decide *which* tool to call and *with what arguments*. The model does not run the tool — it returns `tool_calls` that you execute yourself.

In [10]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")
model_with_tools = model.bind_tools([add, multiply])

response = model_with_tools.invoke("What is 12 multiplied by 9?")

# The model asks to call a tool instead of answering directly
print(response.tool_calls)

[{'name': 'multiply', 'args': {'b': 9, 'a': 12}, 'id': '17a61804-3ee3-4e0f-8cfc-4bfdd64de302', 'type': 'tool_call'}]


In [12]:
# Manually run the tool the model picked
tools_by_name = {"add": add, "multiply": multiply}

for call in response.tool_calls:
    selected = tools_by_name[call["name"]]
    result = selected.invoke(call["args"])
    print(f"{call['name']}({call['args']}) = {result}")

multiply({'b': 9, 'a': 12}) = 108


### 5. Letting an agent use the tools

`create_agent` wires everything together: it calls the model, runs any requested tools, feeds results back, and loops until the model has a final answer.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[add, multiply, get_weather],
    system_prompt="You are a helpful assistant that uses tools to answer questions.",
)

result = agent.invoke(
    {"messages": [("user", "What is 3 plus 4, and what's the weather in Tokyo?")]}
)

# Print the final answer
print(result["messages"][-1].text)

In [ ]:
# See every step the agent took (model messages + tool calls + tool results)
for message in result["messages"]:
    message.pretty_print()